In [4]:
import os
import getpass
import snowflake.connector
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

passcode = getpass.getpass("Snowflake MFA code (from your authenticator app): ")

conn = snowflake.connector.connect(
    user=os.getenv('SNOWFLAKE_USER'),
    password=os.getenv('SNOWFLAKE_PASSWORD'),
    account=os.getenv('SNOWFLAKE_ACCOUNT'),
    warehouse=os.getenv('SNOWFLAKE_WAREHOUSE'),
    database=os.getenv('SNOWFLAKE_DATABASE'),
    schema=os.getenv('SNOWFLAKE_SCHEMA'),
    role=os.getenv('SNOWFLAKE_ROLE'),
    passcode=passcode,
)
cursor = conn.cursor()
print("Connected to Snowflake successfully.")

# 1. Row counts across all marts
print("\n=== ROW COUNTS ===")
for tbl in [
    "SALES_MARTS.MART_ORDER_COMPLETE",
    "SALES_MARTS.MART_CUSTOMER_SUMMARY",
    "SALES_MARTS.MART_ORDER_ITEMS_DETAIL",
    "SALES_STAGING.STG_ORDERS",
    "SALES_STAGING.STG_ORDER_ITEMS",
    "SALES_STAGING.STG_RETURNS",
]:
    cursor.execute(f"SELECT COUNT(*) FROM {tbl}")
    print(f"  {tbl:<45} {cursor.fetchone()[0]:>7} rows")

# 2. Column types for mart_customer_summary (the Boolean issue)
print("\n=== DESCRIBE MART_CUSTOMER_SUMMARY ===")
df_desc = pd.read_sql("DESCRIBE TABLE SALES_MARTS.MART_CUSTOMER_SUMMARY", conn)
df_desc.columns = df_desc.columns.str.lower()
print(df_desc[["name", "type", "null?"]].to_string(index=False))

# 3. Sample of the suspicious column
print("\n=== SAMPLE: TOTAL_ORDERS (first 10 rows) ===")
df_sample = pd.read_sql(
    "SELECT customer_id, total_orders FROM SALES_MARTS.MART_CUSTOMER_SUMMARY LIMIT 10",
    conn
)
df_sample.columns = df_sample.columns.str.lower()
print(df_sample)
print(f"\nDtype in pandas: {df_sample['total_orders'].dtype}")

Connected to Snowflake successfully.

=== ROW COUNTS ===
  SALES_MARTS.MART_ORDER_COMPLETE                 15000 rows
  SALES_MARTS.MART_CUSTOMER_SUMMARY                6200 rows
  SALES_MARTS.MART_ORDER_ITEMS_DETAIL             29668 rows
  SALES_STAGING.STG_ORDERS                        15000 rows
  SALES_STAGING.STG_ORDER_ITEMS                   29668 rows
  SALES_STAGING.STG_RETURNS                        3187 rows

=== DESCRIBE MART_CUSTOMER_SUMMARY ===
                   name              type null?
            CUSTOMER_ID VARCHAR(16777216)     Y
                 REGION VARCHAR(16777216)     Y
                   CITY VARCHAR(16777216)     Y
    ACQUISITION_CHANNEL VARCHAR(16777216)     Y
       CUSTOMER_SEGMENT VARCHAR(16777216)     Y
            SIGNUP_DATE VARCHAR(16777216)     Y
LIFETIME_VALUE_ESTIMATE             FLOAT     Y
                HAS_APP      NUMBER(38,0)     Y
           TOTAL_ORDERS      NUMBER(18,0)     Y
       FIRST_ORDER_DATE  TIMESTAMP_NTZ(9)     Y
        L

C:\Users\frase\AppData\Local\Temp\ipykernel_27228\413552152.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_desc = pd.read_sql("DESCRIBE TABLE SALES_MARTS.MART_CUSTOMER_SUMMARY", conn)
C:\Users\frase\AppData\Local\Temp\ipykernel_27228\413552152.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sample = pd.read_sql(


  customer_id  total_orders
0  CUST-01425             3
1  CUST-03269             2
2  CUST-01339             2
3  CUST-00616             4
4  CUST-03739             6
5  CUST-05939             3
6  CUST-03090             5
7  CUST-00576             5
8  CUST-00460             1
9  CUST-05246             3

Dtype in pandas: int64
